# EDM extension of CLASS — physics plots (Teixeira comparison, kernel, ISW, inner structure)

This notebook rebuilds **classy** from the repository (downloading it from git if needed),
runs the curated `final_ini/`, `final_ini_p6/` models, and reproduces all the relevant
physics plots:

1. **$\Delta\Sigma_8/\Sigma_8$** — our code vs. the Teixeira paper (σ8 scaling).
2. **Kernel $K(\ell)$** — the sign flip at $\ell=2$ (entropy-coupling response).
3. **ISW observable** — low-$\ell$ $C_\ell^{TT}$ residual (ISW-dominated).
4. **Inner structure** — $\Delta\delta_c/\delta_c$, $\Delta\theta_c/\theta_c$, $\Delta\sigma_8/\sigma_8$ vs. redshift (updated Figure 1).
5. **Observables** — $\Delta P(k)/P(k)$, $\Delta C_\ell^{TT}/C_\ell^{TT}$, $\Delta C_\ell^{\phi\phi}/C_\ell^{\phi\phi}$ (updated Figure 2).

Plots are written to `output/plots/`.


In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Alexei117/CLASS_thesis.git"

def ensure_classy():
    try:
        import classy
        print("classy already importable:", classy.__file__)
        return
    except ImportError:
        pass
    # If not running inside the repo root, clone it first.
    if not os.path.exists(os.path.join(os.getcwd(), "source", "perturbations.c")):
        subprocess.run(["git", "clone", REPO_URL, "CLASS_thesis"], check=True)
        os.chdir("CLASS_thesis")
    subprocess.run([sys.executable, "-m", "pip", "install", "."], check=True)
    import classy
    print("classy built & imported:", classy.__file__)

ensure_classy()

import numpy as np
import matplotlib.pyplot as plt
from classy import Class

trapz = getattr(np, "trapezoid", None) or np.trapz

plt.rcParams.update({
    "font.size": 11, "figure.dpi": 120, "savefig.dpi": 300,
    "axes.linewidth": 1.3, "axes.edgecolor": "0.55",
    "axes.grid": True, "grid.color": "0.6", "grid.alpha": 0.25, "grid.linestyle": ":",
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
})
os.makedirs("output/plots", exist_ok=True)
print("setup complete")


In [ ]:
# ---- model-running helpers -------------------------------------------------
STRIP_KEYS = ["root", "write parameters", "write background",
              "phi_ini_scf", "phi_prime_ini_scf"]

def parse_ini(path):
    p = {}
    with open(path) as f:
        for line in f:
            line = line.split("#")[0].strip()
            if not line or "=" not in line:
                continue
            k, v = line.split("=", 1)
            p[k.strip()] = v.strip()
    return p

_cache = {}

def run_ini(path):
    # Run a tracked .ini via classy and cache the Cosmo object.
    path = os.path.normpath(path)
    if path not in _cache:
        p = parse_ini(path)
        for k in STRIP_KEYS:
            p.pop(k, None)
        cosmo = Class()
        cosmo.set(p)
        cosmo.compute()
        _cache[path] = cosmo
    return _cache[path]

def sigma8_at_z(cosmo, R=8.0):
    # sigma8(z) from the linear P(k,z) grid (h units).
    pk, k, z = cosmo.get_pk_and_k_and_z(nonlinear=False, h_units=True)
    s8 = []
    for iz in range(len(z)):
        x = k * R
        W = np.where(x > 0, 3.0 * (np.sin(x) - x * np.cos(x)) / x**3, 1.0)
        s8.append(float(np.sqrt(trapz(k**2 * pk[:, iz] * W**2, k) / (2 * np.pi**2))))
    z = np.asarray(z)
    s8 = np.asarray(s8)
    o = np.argsort(z)
    return z[o], s8[o]

def get_pert(cosmo, k_index=0):
    # Perturbation evolution (a, delta_edm, theta_edm) for the k_index-th k value.
    d = cosmo.get_perturbations()["scalar"][k_index]
    return d["a"], d["delta_edm"], d["theta_edm"]

def get_cl(cosmo, lmax=2500):
    cl = cosmo.lensed_cl(lmax)
    return cl["ell"], cl["tt"], cl["pp"]

def pct_diff(x, y, x0, y0):
    # 100*(y - interp(x0,y0)(x))/interp(x0,y0)(x), NaN where ref is ~0.
    ref = np.interp(x, x0, y0)
    out = np.full(np.shape(y), np.nan)
    ok = np.abs(ref) > 1e-12 * np.nanmax(np.abs(ref))
    out[ok] = 100.0 * (np.asarray(y, float)[ok] - ref[ok]) / ref[ok]
    return out


In [ ]:
# ======================================================================
# 1. Teixeira comparison:  Delta sigma8 / sigma8
# ======================================================================
# p5_paper set maps c01 = 15 * h0 (paper h0). Baseline is c01 = 0.
FINI = "final_ini"
paper_s8 = {+0.01: -1.45, +0.03: -4.30, -0.02: +2.94, -0.04: +5.92}   # Teixeira (%)

signals = [("cpos015", +0.01), ("cp045", +0.03), ("cn030", -0.02), ("cn060", -0.04)]

base = run_ini(f"{FINI}/p5_paper_c0.ini")
s8_base = base.sigma8()

our_h0, our_ds8, pap_ds8 = [], [], []
for name, h0 in signals:
    c = run_ini(f"{FINI}/p5_paper_{name}.ini")
    ds8 = 100.0 * (c.sigma8() - s8_base) / s8_base
    our_h0.append(h0); our_ds8.append(ds8); pap_ds8.append(paper_s8[h0])

our_h0 = np.array(our_h0); our_ds8 = np.array(our_ds8); pap_ds8 = np.array(pap_ds8)

fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.axhline(0, color="k", lw=0.6)
ax.plot(our_h0, our_ds8, "o-", color="#8b3a3a", lw=1.8, ms=6, label=r"our code ($\Delta\sigma_8/\sigma_8$)")
ax.plot(our_h0, pap_ds8, "s--", color="#7fa87f", lw=1.8, ms=6, label=r"Teixeira (paper)")
for h0, a, b in zip(our_h0, our_ds8, pap_ds8):
    ax.plot([h0, h0], [a, b], color="0.6", lw=0.7, zorder=1)
ax.set_xlabel(r"paper $h_0$ offset  (our $c_{01}=15\,h_0$)")
ax.set_ylabel(r"$\Delta\sigma_8/\sigma_8\ \ [\%]$")
ax.set_title(r"Teixeira comparison — $\sigma_8$ scaling (7% systematic)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("output/plots/teixeira_sigma8_comparison.png", bbox_inches="tight")
fig.savefig("output/plots/teixeira_sigma8_comparison.pdf", bbox_inches="tight")
print("our  dS8/S8:", dict(zip(our_h0.tolist(), np.round(our_ds8, 3).tolist())))
print("paper dS8/S8:", dict(zip(our_h0.tolist(), pap_ds8.tolist())))
plt.show()


In [ ]:
# ======================================================================
# 2. Kernel K(l) — sign flip at l = 2
# ======================================================================
# K(l) = [C_l(signal) - C_l(baseline)] / C_l(baseline) / eps,  eps = A_s * c01.
base = run_ini(f"{FINI}/p2_As0p3_c0.ini")      # A_s=0.3, c01=0
sig  = run_ini(f"{FINI}/p2_As0p3_c0p5.ini")    # A_s=0.3, c01=0.5
eps = 0.3 * 0.5

ell0, tt0, _ = get_cl(base)
ell, tt, _    = get_cl(sig)
K = np.full_like(tt, np.nan, dtype=float)
m = tt0 > 1e-30
K[m] = (tt[m] - tt0[m]) / tt0[m] / eps     # dimensionless kernel

fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.axhline(0, color="k", lw=0.6)
ax.axvline(2, color="r", lw=0.6, ls=":", alpha=0.6)
ax.semilogx(ell, K, color="#8b3a3a", lw=1.8)
ax.set_xlim(2, 2500); ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$K(\ell)=\Delta C_\ell^{TT}/C_\ell^{TT}\,/\,\varepsilon$")
ax.set_title(r"Kernel $K(\ell)$ — sign flip at $\ell=2$ (ISW)")

# annotate the low-l structure
for lv in (2, 3, 5):
    i = np.argmin(np.abs(ell - lv))
    ax.annotate(f"$\\ell={lv}$: $K={K[i]:+.2e}$", (ell[i], K[i]),
                textcoords="offset points", xytext=(6, 6 if K[i] > 0 else -12),
                fontsize=9)
fig.tight_layout()
fig.savefig("output/plots/kernel_Kl.png", bbox_inches="tight")
fig.savefig("output/plots/kernel_Kl.pdf", bbox_inches="tight")
for lv in (2, 3, 5, 20):
    i = np.argmin(np.abs(ell - lv))
    print(f"K({lv}) = {K[i]:+.4e}")
plt.show()


In [ ]:
# ======================================================================
# 3. ISW observable — low-l C_l^TT residual
# ======================================================================
# At low l the EDM signal is dominated by the late-ISW response.
ell0, tt0, _ = get_cl(base)
ell, tt, _    = get_cl(sig)
res = np.full_like(tt, np.nan, dtype=float)
m = tt0 > 1e-30
res[m] = 100.0 * (tt[m] - tt0[m]) / tt0[m]

fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.axhline(0, color="k", lw=0.6)
ax.plot(ell, res, color="#5a2a2a", lw=1.8)
ax.set_xlim(2, 100); ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$\Delta C_\ell^{TT}/C_\ell^{TT}\ \ [\%]$")
ax.set_title(r"Low-$\ell$ (ISW) residual — entropy coupling")
ax.axvspan(2, 30, color="0.85", alpha=0.5, zorder=0)
ax.text(0.5, 0.03, "ISW-dominated", transform=ax.transAxes, ha="center",
        fontsize=9, color="0.3")
fig.tight_layout()
fig.savefig("output/plots/isw_lowl.png", bbox_inches="tight")
fig.savefig("output/plots/isw_lowl.pdf", bbox_inches="tight")
print("residual at l=2,3,5:", [float(res[np.argmin(np.abs(ell-lv))]) for lv in (2,3,5)])
plt.show()


In [ ]:
# ======================================================================
# 4. Inner structure — delta_c, theta_c, sigma8 vs redshift (Figure 1, updated)
# ======================================================================
P6 = "final_ini_p6"
# (ini, paper h0, actual c01, style)
RUNS = [
    ("fig1_h0p00.ini",   0.00,  0.000, dict(color="#5a2a2a", ls="--", lw=1.6)),
    ("fig1_h0p01.ini",   0.01,  0.161, dict(color="#8b3a3a", ls="-",  lw=1.8)),
    ("fig1_h0p03.ini",   0.03,  0.483, dict(color="#7fa87f", ls="-.", lw=1.6)),
    ("fig1_hn0p02.ini", -0.02, -0.322, dict(color="#c9a227", ls=":",  lw=1.8)),
    ("fig1_hn0p04.ini", -0.04, -0.644, dict(color="#b89aa0", ls="-",  lw=1.8)),
]

def label(h0, c01):
    return f"$c_{{01}}={c01:+.2f}$  ($h_0={h0:+.2f}$)"

PERT = {}; SIG = {}
for ini, h0, c01, _ in RUNS:
    c = run_ini(f"{P6}/{ini}")
    PERT[h0] = get_pert(c)
    SIG[h0] = sigma8_at_z(c)

a0, d0, t0 = PERT[0.00]
z0, s80 = SIG[0.00]

panels = [[], [], []]
for ini, h0, c01, sty in RUNS:
    a, dc, tc = PERT[h0]
    x = 1.0 / a
    panels[0].append((x, pct_diff(x, dc, 1/a0, d0)))
    panels[1].append((x, pct_diff(x, tc, 1/a0, t0)))
    z, s8 = SIG[h0]
    zg = np.linspace(0.0, min(z.max(), z0.max()), 200)
    panels[2].append((1.0 + zg, pct_diff(1.0+zg, np.interp(zg, z, s8),
                                         1.0+zg, np.interp(zg, z0, s80))))

ylabels = [r"$\Delta\delta_c/\delta_c\ \ [\%]$",
           r"$\Delta\theta_c/\theta_c\ \ [\%]$",
           r"$\Delta\sigma_8/\sigma_8\ \ [\%]$"]

fig, axs = plt.subplots(3, 1, figsize=(7, 9.5), sharex=True)
fig.subplots_adjust(left=0.15, right=0.97, top=0.92, bottom=0.12, hspace=0.0)
for ax, ser, ylab, letter in zip(axs, panels, ylabels, "(a) (b) (c)".split()):
    ax.axhline(0, color="k", lw=0.6)
    for (x, y), (ini, h0, c01, sty) in zip(ser, RUNS):
        ax.plot(x, y, label=label(h0, c01), zorder=3, **sty)
    ax.set_xlim(3.0, 1.0)
    ax.set_ylabel(ylab)
    ax.text(0.015, 0.965, letter, transform=ax.transAxes, fontweight="bold", va="top")
axs[-1].set_xlabel(r"$1+z$")
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, 0.0), columnspacing=1.8)
fig.suptitle(r"Figure 1 — $\delta_c$, $\theta_c$, $\sigma_8$ vs. redshift", y=0.985)
fig.savefig("output/plots/fig1_redshift_updated.png", bbox_inches="tight")
fig.savefig("output/plots/fig1_redshift_updated.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ======================================================================
# 5. Observables — P(k), C_l^TT, C_l^phiphi (Figure 2, updated)
# ======================================================================
base = run_ini(f"{P6}/fig1_h0p00.ini")
pk0, k0, z0v = base.get_pk_and_k_and_z(nonlinear=False, h_units=True)
i0 = np.argmin(np.abs(z0v - 0.0))
ell0, tt0, pp0 = get_cl(base)

curves = [[], [], []]
for ini, h0, c01, sty in RUNS:
    c = run_ini(f"{P6}/{ini}")
    pk, k, zv = c.get_pk_and_k_and_z(nonlinear=False, h_units=True)
    ii = np.argmin(np.abs(zv - 0.0))
    curves[0].append((k, pct_diff(k, pk[:, ii], k0, pk0[:, i0])))
    ell, tt, pp = get_cl(c)
    curves[1].append((ell, pct_diff(ell, tt, ell0, tt0)))
    curves[2].append((ell, pct_diff(ell, pp, ell0, pp0)))

ylabels = [r"$\Delta P(k)/P(k)\ \ [\%]$",
           r"$\Delta C_\ell^{TT}/C_\ell^{TT}\ \ [\%]$",
           r"$\Delta C_\ell^{\phi\phi}/C_\ell^{\phi\phi}\ \ [\%]$"]

fig, axs = plt.subplots(3, 1, figsize=(7, 9.5))
fig.subplots_adjust(left=0.15, right=0.97, top=0.92, bottom=0.12, hspace=0.28)
for ax, cv, ylab, letter in zip(axs, curves, ylabels, "(a) (b) (c)".split()):
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xscale("log")
    for (x, y), (ini, h0, c01, sty) in zip(cv, RUNS):
        ax.plot(x, y, label=label(h0, c01), zorder=3, **sty)
    ax.set_ylabel(ylab)
    ax.text(0.02, 0.965, letter, transform=ax.transAxes, fontweight="bold", va="top")
axs[0].set_xlim(1e-4, 1.0); axs[0].set_xlabel(r"$k\ \ [h\,\mathrm{Mpc}^{-1}]$")
axs[1].set_xlim(2, 2500); axs[2].set_xlim(2, 2500)
axs[1].set_xlabel(r"$\ell$"); axs[2].set_xlabel(r"$\ell$")
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, 0.0), columnspacing=1.8)
fig.suptitle(r"Figure 2 — $P(k)$, $C_\ell^{TT}$, $C_\ell^{\phi\phi}$", y=0.985)
fig.savefig("output/plots/fig2_pk_cl_updated.png", bbox_inches="tight")
fig.savefig("output/plots/fig2_pk_cl_updated.pdf", bbox_inches="tight")
plt.show()


Done. Plots are in `output/plots/`.